# Differential Abundance of Malignant States Between Treatment Stages

Tests differential abundance of malignant-cell neighbourhoods (milopy, edgeR
NB-GLM with graph-weighted spatial FDR) for three pairwise treatment-stage
comparisons, adjusting for smoking status:

- treatment-naive vs on-treatment
- treatment-naive vs TKI-resistant
- on-treatment vs TKI-resistant

For each comparison, logFC > 0 means the neighbourhood is enriched in the second stage.
Results are plotted as neighbourhood graphs and miloR-style DA beeswarms
(`plotDAbeeswarm` equivalent), replacing the separate R plotting notebook (1.6b).

In [ ]:
# R + edgeR are required by milopy (called through rpy2)
# !apt-get install -y r-base r-base-dev
# !R -e "install.packages(c('BiocManager', 'statmod'), repos='http://cran.rstudio.com/')"
# !R -e "BiocManager::install('edgeR', ask=FALSE, update=FALSE)"
# !pip install --quiet scanpy rpy2 git+https://github.com/emdann/milopy.git@30646f538481151b6101b5e2f133858d2171000a
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import os
import warnings

import importlib.util
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore", category=FutureWarning)

# milopy (last release, unmaintained) breaks on recent scipy/pandas: it indexes a sparse
# matrix with a pandas Series and writes floats into an int column. Idempotent source
# patch, applied before import; no-op on older environments where the lines differ.
_core = pathlib.Path(importlib.util.find_spec("milopy").submodule_search_locations[0]) / "core.py"
_src = _core.read_text()
for _old, _new in [
    ('nhood_ixs = adata.obs["nhood_ixs_refined"] == 1\n',
     'nhood_ixs = (adata.obs["nhood_ixs_refined"] == 1).to_numpy()\n'),
    ('adata.obs["nhood_kth_distance"] = 0\n',
     'adata.obs["nhood_kth_distance"] = 0.0\n'),
]:
    if _old in _src:
        _src = _src.replace(_old, _new)
        print("patched milopy.core:", _new.strip())
_core.write_text(_src)

import milopy.core as milo
import milopy.utils

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join.h5ad"
milo_output_dir = "./Milo"
figures_output_dir = "./figures/milo"

base_label_col = "cell_type_new"
malignant_label = "Malignant"
state_col = "final_state"
unassigned_label = "unassigned"
recist_col = "recist_v3"
sample_col = "ID"                       # one sample per ID; Milo counts cells per nhood per sample

# covariate adjusted for in the design (~ covariate + Condition)
covariate = "smoking_status_grp"
drop_samples_with_label = "unknown"     # samples with this covariate value are dropped; None to keep as a level

# pairwise comparisons: (reference, test); logFC > 0 = enriched in test
comparisons = {
    "TN_vs_OT": ("1_treatment_naive", "3_on_treatment"),
    "TN_vs_R": ("1_treatment_naive", "4_TKI_resistance"),
    "OT_vs_R": ("3_on_treatment", "4_TKI_resistance"),
}

# neighbourhood construction
use_rep = "X_scVI_n_latent_hvg_10"
n_neighbors = 100
prop = 0.15                             # fraction of cells sampled as nhood index candidates
seed = 0

# results
spatial_fdr_alpha = 0.15                # SpatialFDR threshold for colouring / counting significant nhoods
mixed_threshold = 0.5                   # nhoods whose majority state is < this fraction -> "Mixed"
nhood_graph_basis = "X_draw_graph_fa"   # layout for the nhood graph plot

# DA beeswarm (miloR plotDAbeeswarm equivalent)
beeswarm_exclude_mixed = True
beeswarm_logfc_limits = (-9, 9)         # as in 1.4c; nhoods outside are not shown (count printed). None = fit data
beeswarm_single_figsize = (7, 5)
# ggplot2 scales::muted("blue") / "white" / scales::muted("red")
da_cmap = LinearSegmentedColormap.from_list("milo_da", ["#3A3A98", "#FFFFFF", "#832424"])

state_order = [
    "1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
    "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative", "Mixed",
]
state_colors = {
    "1_Epithelial baseline": "#8dd3c7",
    "2_Differentiated epithelial": "#bebada",
    "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5",
    "5_Neurod_EMT-like": "#bc80bd",
    "6_Proliferative": "#ffed6f",
    "Mixed": "lightgrey",
}

os.makedirs(milo_output_dir, exist_ok=True)
os.makedirs(figures_output_dir, exist_ok=True)

## 1. Load malignant cells

In [ ]:
adata = sc.read(input_path)
adata = adata[(adata.obs[base_label_col] == malignant_label)
              & (adata.obs[state_col] != unassigned_label)].copy()

if drop_samples_with_label is not None:
    n_before = adata.obs[sample_col].nunique()
    adata = adata[adata.obs[covariate].astype(str) != drop_samples_with_label].copy()
    print(f"dropped {n_before - adata.obs[sample_col].nunique()} samples with {covariate} = {drop_samples_with_label}")

adata.obs[state_col].value_counts()

## 2. Design check

Milo fits one model per comparison at sample level, so the covariate × condition
table that matters is the number of **samples**, not cells. A covariate level present
in only one condition is partly confounded with it and weakens the adjustment.

In [ ]:
design_tables = {}
for name, (ref, test) in comparisons.items():
    obs = adata.obs[adata.obs[recist_col].isin([ref, test])]
    samples = obs.groupby(sample_col, observed=True)[[covariate, recist_col]].first()
    ambiguous = obs.groupby(sample_col, observed=True)[[covariate, recist_col]].nunique().max()
    assert (ambiguous <= 1).all(), f"{name}: a sample maps to more than one {covariate}/{recist_col} value"
    table = pd.crosstab(samples[covariate], samples[recist_col]).reindex(columns=[ref, test], fill_value=0)
    design_tables[name] = table
    print(f"--- {name}: samples per {covariate} x condition")
    print(table, "\n")
    if (table == 0).any(axis=None):
        print(f"WARNING {name}: some {covariate} levels are absent from one condition\n")

## 3. Run Milo per comparison

For each comparison: subset to the two stages, rebuild the kNN graph on the scVI
latent space, sample neighbourhoods, count cells per sample, and test
`~ covariate + Condition` (the Condition coefficient is tested; its reference level
is the first stage of the pair). Neighbourhoods are annotated by majority state.

In [ ]:
def run_milo(ad, ref, test):
    sub = ad[ad.obs[recist_col].isin([ref, test])].copy()
    sub.obs["Condition"] = pd.Categorical(sub.obs[recist_col].astype(str), categories=[ref, test])
    sub.obs[covariate] = sub.obs[covariate].astype(str).astype("category")

    sc.pp.neighbors(sub, use_rep=use_rep, n_neighbors=n_neighbors, key_added="milo", random_state=seed)
    np.random.seed(seed)
    milo.make_nhoods(sub, neighbors_key="milo", prop=prop)
    milo.count_nhoods(sub, sample_col=sample_col)
    milo.DA_nhoods(sub, design=f"~{covariate}+Condition")      # tests the last coefficient (Condition)

    milopy.utils.annotate_nhoods(sub, anno_col=state_col)
    milopy.utils.build_nhood_graph(sub, basis=nhood_graph_basis)
    res = sub.uns["nhood_adata"].obs
    res["nhood_annotation"] = res["nhood_annotation"].astype(str)
    res.loc[res["nhood_annotation_frac"] < mixed_threshold, "nhood_annotation"] = "Mixed"
    res["nhood_annotation"] = pd.Categorical(res["nhood_annotation"], categories=state_order)
    return sub

milo_results = {}
for name, (ref, test) in comparisons.items():
    print(f"=== {name}: {ref} (ref) vs {test}")
    sub = run_milo(adata, ref, test)
    res = sub.uns["nhood_adata"].obs
    res.to_csv(f"{milo_output_dir}/nhood_milo_malignant_{name}_{covariate}_prop{prop}.csv")
    milo_results[name] = sub
    n_sig = (res["SpatialFDR"] < spatial_fdr_alpha).sum()
    print(f"{len(res)} nhoods, {n_sig} with SpatialFDR < {spatial_fdr_alpha} "
          f"({(res.loc[res.SpatialFDR < spatial_fdr_alpha, 'logFC'] > 0).sum()} up in {test})\n")

### QC: neighbourhood sizes, p-value distributions and volcano plots

In [ ]:
fig, axes = plt.subplots(3, len(comparisons), figsize=(4.5 * len(comparisons), 10))
for c, (name, sub) in enumerate(milo_results.items()):
    res = sub.uns["nhood_adata"].obs
    axes[0, c].hist(np.asarray(sub.obsm["nhoods"].sum(0)).ravel(), bins=100)
    axes[0, c].set(title=name, xlabel="Nhood size (cells)")
    axes[1, c].hist(res["PValue"], bins=50)
    axes[1, c].set(xlabel="P-value")
    axes[2, c].scatter(res["logFC"], -np.log10(res["SpatialFDR"]), s=3)
    axes[2, c].axhline(-np.log10(spatial_fdr_alpha), ls="--", c="grey", lw=0.8)
    axes[2, c].set(xlabel="logFC", ylabel="-log10(SpatialFDR)")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/milo_qc.pdf", bbox_inches="tight")
plt.show()

## 4. Neighbourhood graphs

Nodes are neighbourhoods at the FDG position of their index cell, sized by number of
cells; colour is logFC for neighbourhoods with SpatialFDR < alpha (grey otherwise).
All three panels share one symmetric colour scale.

In [ ]:
vmax = max(sub.uns["nhood_adata"].obs.loc[sub.uns["nhood_adata"].obs.SpatialFDR < spatial_fdr_alpha, "logFC"].abs().max()
           for sub in milo_results.values())
vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1

fig, axes = plt.subplots(1, len(comparisons), figsize=(5.5 * len(comparisons), 5))
for ax, (name, sub) in zip(axes, milo_results.items()):
    nh = sub.uns["nhood_adata"]
    res, pos = nh.obs, nh.obsm["X_milo_graph"]
    sig = (res["SpatialFDR"] < spatial_fdr_alpha).values
    size = res["Nhood_size"].values * 0.5
    order = np.argsort(np.where(sig, res["logFC"].abs().values, -1))
    ax.scatter(pos[order][~sig[order], 0], pos[order][~sig[order], 1], s=size[order][~sig[order]],
               c="lightgrey", linewidths=0, rasterized=True)
    sc_ = ax.scatter(pos[order][sig[order], 0], pos[order][sig[order], 1], s=size[order][sig[order]],
                     c=res["logFC"].values[order][sig[order]], cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                     linewidths=0, rasterized=True)
    ref, test = comparisons[name]
    ax.set_title(f"{ref} → {test}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.colorbar(sc_, ax=axes, label="logFC (SpatialFDR < %.2g)" % spatial_fdr_alpha, shrink=0.7)
plt.savefig(f"{figures_output_dir}/milo_nhood_graph_3comparisons.pdf", dpi=200, bbox_inches="tight")
plt.show()

## 5. DA beeswarm by cell state

Two-layer beeswarm matching `1_4c_plot_milo_DAtest__malignant.R`: non-significant
neighbourhoods (SpatialFDR ≥ alpha) small, light grey and transparent underneath;
significant ones larger and coloured by logFC (muted blue–white–red, centred at 0) on
top. Point spread is density-scaled quasi-random (approximating `geom_beeswarm` with
`priority = "density"`); the first state is at the bottom. All panels share one
symmetric colour scale.

In [ ]:
def _van_der_corput(n, base=2):
    seq = np.zeros(n)
    for i in range(n):
        f, x, k = 1.0, 0.0, i + 1
        while k:
            f /= base
            x += f * (k % base)
            k //= base
        seq[i] = x
    return seq

def quasirandom_offsets(values, width):
    """Offsets in [-width, width] scaled by local density of `values`."""
    n = len(values)
    if n < 3 or np.std(values) == 0:
        return np.zeros(n)
    density = gaussian_kde(values)(values)
    density = density / density.max()
    order = np.argsort(values)
    offsets = np.empty(n)
    offsets[order] = (_van_der_corput(n) - 0.5) * 2
    return offsets * density * width

def plot_da_beeswarm(res, ax, title, vmax, alpha=spatial_fdr_alpha, limits=beeswarm_logfc_limits):
    res = res.copy()
    res["nhood_annotation"] = res["nhood_annotation"].astype(str)
    if beeswarm_exclude_mixed:
        res = res[res["nhood_annotation"] != "Mixed"]
    if limits is not None:
        outside = ~res["logFC"].between(*limits)
        if outside.any():
            print(f"{title}: {outside.sum()} nhoods outside logFC {limits} not shown "
                  f"({(outside & (res['SpatialFDR'] < alpha)).sum()} significant)")
        res = res[~outside]
    groups = [s for s in state_order if s in set(res["nhood_annotation"])]
    pts = None
    for y, grp in enumerate(groups):
        d = res[res["nhood_annotation"] == grp]
        sig = (d["SpatialFDR"] < alpha).values
        ns_lfc, sig_lfc = d["logFC"].values[~sig], d["logFC"].values[sig]
        ax.scatter(ns_lfc, y + quasirandom_offsets(ns_lfc, 0.15), s=2, c="#CCCCCC", alpha=0.1,
                   linewidths=0, rasterized=True, zorder=1)
        pts = ax.scatter(sig_lfc, y + quasirandom_offsets(sig_lfc, 0.4), s=12, c=sig_lfc, cmap=da_cmap,
                         vmin=-vmax, vmax=vmax, linewidths=0, rasterized=True, zorder=2)
    ax.axvline(0, ls="--", color="black", lw=0.8, zorder=0)
    ax.set_yticks(range(len(groups)), groups, fontsize=10)
    ax.set_xlabel("Log Fold Change")
    ax.set_title(title, fontsize=9)
    if limits is not None:
        ax.set_xlim(*limits)
    return pts

all_res = pd.concat([sub.uns["nhood_adata"].obs.assign(comparison=name) for name, sub in milo_results.items()])
sig_lfc = all_res.loc[all_res["SpatialFDR"] < spatial_fdr_alpha, "logFC"].abs()
da_vmax = sig_lfc.max() if len(sig_lfc) else 1.0

# one PDF per comparison (as in 1.4c)
for name, (ref, test) in comparisons.items():
    fig, ax = plt.subplots(figsize=beeswarm_single_figsize)
    pts = plot_da_beeswarm(all_res[all_res["comparison"] == name], ax, f"{ref} → {test}", da_vmax)
    ax.set_ylabel("Nhood annotation")
    if pts is not None:
        fig.colorbar(pts, ax=ax, label="logFC")
    plt.tight_layout()
    plt.savefig(f"{figures_output_dir}/nhood_milo_malignant_{name}_beeswarm_FDR{spatial_fdr_alpha}_{covariate}.pdf",
                dpi=200, bbox_inches="tight")
    plt.show()

# combined 1 x 3 panel
fig, axes = plt.subplots(1, len(comparisons), figsize=(5 * len(comparisons), 4), sharey=True)
for ax, (name, (ref, test)) in zip(axes, comparisons.items()):
    pts = plot_da_beeswarm(all_res[all_res["comparison"] == name], ax, f"{ref} → {test}", da_vmax)
axes[0].set_ylabel("Nhood annotation")
fig.colorbar(pts, ax=axes, label=f"logFC (SpatialFDR < {spatial_fdr_alpha})", shrink=0.8)
plt.savefig(f"{figures_output_dir}/milo_DA_beeswarm_3comparisons.pdf", dpi=200, bbox_inches="tight")
plt.show()

### Summary: significant neighbourhoods per state and direction

In [ ]:
summary = (all_res.assign(direction=np.where(all_res["logFC"] > 0, "up_in_test", "up_in_ref"),
                          significant=all_res["SpatialFDR"] < spatial_fdr_alpha)
           .groupby(["comparison", "nhood_annotation", "direction"], observed=True)["significant"]
           .agg(n_nhoods="size", n_significant="sum")
           .reset_index())
summary.to_csv(f"{milo_output_dir}/milo_summary_by_state_3comparisons.csv", index=False)
summary.pivot_table(index="nhood_annotation", columns=["comparison", "direction"],
                    values="n_significant", fill_value=0, observed=True)